# Configuración de paths e imports


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.append(str(PROJECT_ROOT))

from dataclasses import asdict
import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np

from Code.image import ImgPreprocDebugger, ImgPreprocCfg

intento_numero = 9
index = 1
labels = ["Tornillo", "Tuerca", "Arandela", "Clavo"]

guardar = True
batch_mode = True
mostrar = True
mostrar_stages = True


# Configuración del preprocesador


In [2]:
cfg = ImgPreprocCfg(
    target_size=(256, 256),
    clahe_clip=1.0,
    clahe_grid=(20, 20),
    filter_mode="guided",
    bilateral_d=9,
    bilateral_sigma_color=75,
    bilateral_sigma_space=75,
    canny_ratio=0.55,
    detect_edge_mode="percentile",
    morph_kernel_size=(5, 5),
)
debugger = ImgPreprocDebugger(cfg)


# Ejecución del pipeline y visualización


In [ ]:
if not batch_mode:
    categorias = ["Tuerca"]
else:
    categorias = labels

output_root = PROJECT_ROOT / "DataBase" / "tmp" / f"ImgPreprocDebug{intento_numero}"
output_root.mkdir(parents=True, exist_ok=True)

def _create_pipeline_figure(debugger, stage_pairs):
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    if not isinstance(axes, np.ndarray):
        axes = np.array([axes])
    axes = axes.flatten()
    for ax, (title, value) in zip(axes, stage_pairs):
        if isinstance(value, np.ndarray):
            try:
                debugger._imshow(value, title=title, ax=ax)
                continue
            except TypeError:
                ax.set_title(title)
                ax.axis("off")
        else:
            ax.set_title(title)
            ax.axis("off")
    for ax in axes[len(stage_pairs):]:
        ax.axis("off")
    fig.tight_layout()
    return fig

for categoria in categorias:
    categoria_dir = output_root / categoria
    categoria_dir.mkdir(parents=True, exist_ok=True)

    config_path = categoria_dir / "config.txt"
    with config_path.open("w", encoding="utf-8") as fh:
        for key, value in asdict(cfg).items():
            fh.write(f"{key} = {value}\n")

    for i in range(index):
        img_path = PROJECT_ROOT / "DataBase" / "data" / "images1" / categoria / f"{categoria}{i+1:03d}.jpg"
        img_bgr = cv.imread(str(img_path))
        if img_bgr is None:
            raise FileNotFoundError(f"No pude leer {img_path}")

        try:
            cropped, mask, meta, stages = debugger.process(img_path)
        except ValueError as exc:
            print(f"[WARN] {img_path.name}: {exc}")
            continue

        mask_stage = stages.get("mask")
        if not isinstance(mask_stage, np.ndarray):
            mask_stage = np.zeros(img_bgr.shape[:2], dtype=np.uint8)

        cropped_display = cropped if isinstance(cropped, np.ndarray) else stages.get("cropped_img")
        if not isinstance(cropped_display, np.ndarray):
            cropped_display = img_bgr

        cropped_mask_display = mask if isinstance(mask, np.ndarray) else stages.get("cropped_mask")
        if not isinstance(cropped_mask_display, np.ndarray):
            target_h, target_w = cfg.target_size
            cropped_mask_display = np.zeros((target_h, target_w), dtype=np.uint8)

        stage_pairs = [
            ("Img original", img_bgr),
            ("Lab", stages.get("lab")),
            ("L_eq", stages.get("L_eq")),
            ("L_filtrado", stages.get("L_filtered")),
            ("Canny", stages.get("edges")),
            ("Mask", mask_stage),
            ("Cropped", cropped_display),
            ("Cropped mask", cropped_mask_display),
        ]

        pipeline_fig = None
        if guardar or mostrar or mostrar_stages:
            pipeline_fig = _create_pipeline_figure(debugger, stage_pairs)

        if guardar:
            cv.imwrite(str(categoria_dir / img_path.name), cropped if cropped is not None else img_bgr)
            mask_to_save = mask if mask is not None else stages.get("mask")
            if mask_to_save is None:
                mask_to_save = np.zeros(img_bgr.shape[:2], dtype=np.uint8)
            cv.imwrite(str(categoria_dir / f"{img_path.stem}_mask.png"), mask_to_save)
            if pipeline_fig is not None:
                pipeline_path = categoria_dir / f"{img_path.stem}_pipeline.png"
                pipeline_fig.savefig(str(pipeline_path))

        if mostrar and pipeline_fig is not None:
            plt.show()

        if pipeline_fig is not None:
            plt.close(pipeline_fig)

